# Test MedGemma sur nos 20 images

On fait tourner MedGemma sur les 20 images du projet puis on compare avec la vraie classe (déduite du nom du fichier), et on regarde les erreurs.

Avant de lancer :
1. Aller sur https://huggingface.co/google/medgemma-4b-it et accepter l'accès si pas déjà fait
2. Ne pas mettre le token en clair dans le code, le notebook le demande directement
3. Dans Google Colab : Exécution > Modifier le type d'exécution > GPU (T4)



## 1. Installation des librairies

In [2]:
!pip install -q transformers accelerate pillow huggingface_hub pandas

## 2. Connexion sécurisée à Hugging Face



In [3]:
from huggingface_hub import login
login()

## 3. Récupérer les 20 images depuis GitHub

In [4]:
!git clone https://github.com/jenniferrkt/assistant-radiologue-virtuel.git repo
%cd repo

from pathlib import Path

images = sorted(Path("images").glob("*.png")) + sorted(Path("images").glob("*.jpg"))
images = [p for p in images if "assets" not in str(p)]
print(f"{len(images)} images trouvées :")
for p in images:
    print(" -", p.name)

Cloning into 'repo'...
remote: Enumerating objects: 199, done.
remote: Counting objects: 100% (199/199), done.
remote: Compressing objects: 100% (166/166), done.
remote: Total 199 (delta 53), reused 150 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (199/199), 8.50 MiB | 39.74 MiB/s, done.
Resolving deltas: 100% (53/53), done.
/content/repo
20 images trouvées :
 - image01_normal.png
 - image01_suspected_opacity.png
 - image01_uncertain.png
 - image02_normal.png
 - image02_suspected_opacity.png
 - image02_uncertain.png
 - image03_normal.png
 - image03_suspected_opacity.png
 - image03_uncertain.png
 - image04_normal.png
 - image04_suspected_opacity.png
 - image04_uncertain.png
 - image05_normal.png
 - image05_suspected opacity.png
 - image05_uncertain.png
 - image06_normal.png
 - image06_suspected_opacity.png
 - image06_uncertain.png
 - image07_normal.png
 - image07_suspected_opacity.png


## 4. Déduire la vraie étiquette (ground truth) depuis le nom du fichier

In [5]:
def label_from_filename(path: Path) -> str:
    name = path.stem.lower().replace(" ", "_")
    if "suspected" in name or "opacity" in name:
        return "suspected_opacity"
    if "uncertain" in name:
        return "uncertain"
    if "normal" in name:
        return "normal"
    return "unknown"

for p in images:
    print(f"{p.name:45s} -> {label_from_filename(p)}")

image01_normal.png                            -> normal
image01_suspected_opacity.png                 -> suspected_opacity
image01_uncertain.png                         -> uncertain
image02_normal.png                            -> normal
image02_suspected_opacity.png                 -> suspected_opacity
image02_uncertain.png                         -> uncertain
image03_normal.png                            -> normal
image03_suspected_opacity.png                 -> suspected_opacity
image03_uncertain.png                         -> uncertain
image04_normal.png                            -> normal
image04_suspected_opacity.png                 -> suspected_opacity
image04_uncertain.png                         -> uncertain
image05_normal.png                            -> normal
image05_suspected opacity.png                 -> suspected_opacity
image05_uncertain.png                         -> uncertain
image06_normal.png                            -> normal
image06_suspected_opacity.png     

Vérifier la liste au-dessus. Si une classe est fausse, la corriger dans la cellule d'après.

In [6]:

MANUAL_OVERRIDES = {
    # "nom_du_fichier.png": "normal" | "suspected_opacity" | "uncertain",
}

def get_ground_truth(path: Path) -> str:
    if path.name in MANUAL_OVERRIDES:
        return MANUAL_OVERRIDES[path.name]
    return label_from_filename(path)

## 5. Charger le modèle MedGemma (peut prendre quelques minutes)

In [7]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/medgemma-4b-it"

print("Chargement de MedGemma 4B... (peut prendre 3-5 min)")
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
)
print(f"Modèle chargé sur : {next(model.parameters()).device}")

Chargement de MedGemma 4B... (peut prendre 3-5 min)


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

Modèle chargé sur : cuda:0


## 6. Charger le prompt du projet

On utilise `prompts/improved_prompt.txt`dans le dépôt github

In [8]:
with open("prompts/improved_prompt.txt", "r", encoding="utf-8") as f:
    PROMPT_TEXT = f.read()

print(PROMPT_TEXT)

You are a virtual radiology assistant for educational purposes only.

Analyze this chest X-ray step by step:
STEP 1 - Image quality: well exposed? full inspiration? rotation?
STEP 2 - Lung fields: any opacity, consolidation, effusion?
STEP 3 - Cardiac silhouette: normal size?
STEP 4 - Costophrenic angles: sharp or blunted?
STEP 5 - Classify based on steps above.

Then return ONLY this JSON (no extra text):
{
  "predicted_class": "<normal | suspected_opacity | uncertain>",
  "confidence": <float 0.0-1.0>,
  "visual_evidence": "<findings from steps 1-4>",
  "justification": "<reasoning linking findings to class>",
  "limitations": "<quality or ambiguity issues>",
  "warning": "AI prototype for educational use only. Not a medical device. Consult a radiologist."
}
Rule: if confidence < 0.60 → predicted_class must be uncertain.



## 7. Fonctions pour faire tourner le modèle

In [9]:
import json
import re
from PIL import Image

def extract_json(text: str):
    text = text.strip()
    text = re.sub(r"^```json\s*|```$", "", text, flags=re.MULTILINE).strip()
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def predict_one(image_path: Path) -> dict:
    image = Image.open(image_path).convert("RGB")

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": PROMPT_TEXT},
        ],
    }]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device, dtype=torch.bfloat16)

    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=300, do_sample=False)

    decoded = processor.decode(
        output[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )

    parsed = extract_json(decoded)
    if parsed is None:
        return {"predicted_class": "uncertain", "confidence": 0.0, "raw_output": decoded, "parse_ok": False}
    parsed["raw_output"] = decoded
    parsed["parse_ok"] = True
    return parsed

## 8. Lancer les prédictions sur les 20 images


In [14]:
import pandas as pd
import re

HEDGE_WORDS = ["could be", "possible", "suggests", "raises suspicion",
               "further evaluation", "difficult to", "may represent",
               "cannot rule out", "concerning"]

rows = []

for i, img_path in enumerate(images):
    ground_truth = get_ground_truth(img_path)
    case_id = f"RSNA_{i+1:03d}"

    print(f"[{i+1}/{len(images)}] {img_path.name} (true: {ground_truth}) ...", end=" ")

    try:
        pred = predict_one(img_path)
        prediction = pred.get("predicted_class", "uncertain")
        print(f"-> {prediction}")
    except Exception as e:
        prediction = "uncertain"
        pred = {"raw_output": f"ERROR: {e}", "parse_ok": False, "justification": ""}
        print(f"-> ERROR: {e}")

    is_correct = prediction == ground_truth
    justification = pred.get("justification", "") or ""
    has_hedge = any(w in justification.lower() for w in HEDGE_WORDS)

    # type d'erreur automatique
    if is_correct:
        error_type = ""
    elif prediction == "uncertain":
        error_type = "UA"
    elif ground_truth == "normal" and prediction == "suspected_opacity":
        error_type = "FP"
    elif ground_truth == "suspected_opacity" and prediction == "normal":
        error_type = "FN"
    else:
        error_type = "other"

    # sévérité automatique
    if is_correct:
        severity = ""
    elif ground_truth == "uncertain" and prediction == "normal":
        severity = "high"
    elif ground_truth == "normal" and prediction != "normal":
        severity = "medium"
    elif has_hedge:
        severity = "medium"
    else:
        severity = "low"

    # commentaire automatique
    if is_correct:
        comment = ""
    elif has_hedge and prediction != "uncertain":
        comment = "model hedged in text but still picked a class"
    elif ground_truth == "normal" and prediction != "normal":
        comment = "model flagged an issue on an image that is actually normal"
    elif prediction == "uncertain":
        comment = "model played safe given image quality or unclear signs"
    else:
        comment = "model was confident but got it wrong"

    # piste d'amélioration automatique
    if is_correct:
        corrective_action = ""
    elif has_hedge and prediction != "uncertain":
        corrective_action = "force uncertain when text shows doubt"
    elif ground_truth == "normal" and prediction != "normal":
        corrective_action = "review sensitivity; check why normal cases get flagged"
    elif prediction == "uncertain":
        corrective_action = "no urgent fix; reasonable caution"
    else:
        corrective_action = "review this case manually"

    rows.append({
        "case_id": case_id,
        "image": img_path.name,
        "ground_truth": ground_truth,
        "prediction": prediction,
        "correct": is_correct,
        "error_type": error_type,
        "severity": severity,
        "comment": comment,
        "corrective_action": corrective_action,
        "justification_modele": justification,
    })

results_df = pd.DataFrame(rows)
results_df

[1/20] image01_normal.png (true: normal) ... -> normal
[2/20] image01_suspected_opacity.png (true: suspected_opacity) ... -> suspected_opacity
[3/20] image01_uncertain.png (true: uncertain) ... -> suspected_opacity
[4/20] image02_normal.png (true: normal) ... -> suspected_opacity
[5/20] image02_suspected_opacity.png (true: suspected_opacity) ... -> suspected_opacity
[6/20] image02_uncertain.png (true: uncertain) ... -> suspected_opacity
[7/20] image03_normal.png (true: normal) ... -> normal
[8/20] image03_suspected_opacity.png (true: suspected_opacity) ... -> suspected_opacity
[9/20] image03_uncertain.png (true: uncertain) ... -> suspected_opacity
[10/20] image04_normal.png (true: normal) ... -> normal
[11/20] image04_suspected_opacity.png (true: suspected_opacity) ... -> suspected_opacity
[12/20] image04_uncertain.png (true: uncertain) ... -> suspected_opacity
[13/20] image05_normal.png (true: normal) ... -> normal
[14/20] image05_suspected opacity.png (true: suspected_opacity) ... ->

,case_id,image,ground_truth,prediction,correct,error_type,severity,comment,corrective_action,justification_modele
0,RSNA_001,image01_normal.png,normal,normal,True,,,,,"The image quality is good, and there are no ap..."
1,RSNA_002,image01_suspected_opacity.png,suspected_opacity,suspected_opacity,True,,,,,The presence of significant opacity in the low...
2,RSNA_003,image01_uncertain.png,uncertain,suspected_opacity,False,other,medium,model hedged in text but still picked a class,force uncertain when text shows doubt,The presence of significant opacity in the rig...
3,RSNA_004,image02_normal.png,normal,suspected_opacity,False,FP,medium,model hedged in text but still picked a class,force uncertain when text shows doubt,The hazy opacities in the right lung field are...
4,RSNA_005,image02_suspected_opacity.png,suspected_opacity,suspected_opacity,True,,,,,The presence of bilateral opacities in the lun...
5,RSNA_006,image02_uncertain.png,uncertain,suspected_opacity,False,other,medium,model hedged in text but still picked a class,force uncertain when text shows doubt,The increased opacity in the right upper lung ...
6,RSNA_007,image03_normal.png,normal,normal,True,,,,,"The image quality is good, and there are no ap..."
7,RSNA_008,image03_suspected_opacity.png,suspected_opacity,suspected_opacity,True,,,,,The diffuse bilateral opacities are highly sug...
8,RSNA_009,image03_uncertain.png,uncertain,suspected_opacity,False,other,medium,model hedged in text but still picked a class,force uncertain when text shows doubt,The significant opacity in the right lung fiel...
9,RSNA_010,image04_normal.png,normal,normal,True,,,,,"Based on the clear lung fields, normal cardiac..."


## 9. Résumé des résultats


In [15]:
n_total = len(results_df)
n_correct = results_df["correct"].sum()

print(f"Total : {n_total} images")
print(f"Corrects : {n_correct} ({round(100*n_correct/n_total,1)}%)")
print(f"Erreurs à analyser : {n_total - n_correct}")
print()
print("Cas à documenter dans le registre d'erreurs :")
display(results_df[~results_df["correct"]][["case_id", "image", "ground_truth", "prediction", "error_type"]])

Total : 20 images
Corrects : 12 (60.0%)
Erreurs à analyser : 8

Cas à documenter dans le registre d'erreurs :


,case_id,image,ground_truth,prediction,error_type
2,RSNA_003,image01_uncertain.png,uncertain,suspected_opacity,other
3,RSNA_004,image02_normal.png,normal,suspected_opacity,FP
5,RSNA_006,image02_uncertain.png,uncertain,suspected_opacity,other
8,RSNA_009,image03_uncertain.png,uncertain,suspected_opacity,other
11,RSNA_012,image04_uncertain.png,uncertain,suspected_opacity,other
14,RSNA_015,image05_uncertain.png,uncertain,suspected_opacity,other
17,RSNA_018,image06_uncertain.png,uncertain,normal,other
18,RSNA_019,image07_normal.png,normal,uncertain,UA


## 10. Télécharger le CSV

Pour chaque ligne en erreur, remplir :
- severity (low/medium/high)
- comment (pourquoi le modèle s'est trompé)
- corrective_action (une idée pour améliorer)



In [16]:
from google.colab import files

final_df = results_df[~results_df["correct"]][
    ["case_id", "ground_truth", "prediction", "error_type", "severity", "comment", "corrective_action"]
]
final_df.to_csv("error_register_final.csv", index=False)

print(f"{len(final_df)} erreurs sur {len(results_df)} images")
files.download("error_register_final.csv")

8 erreurs sur 20 images


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Ce que ça veut vraiment dire

12/20 corrects, 8 erreurs. Mais 6 des 8 erreurs suivent le même schéma : le modèle hésite dans son texte ("could be", "raises suspicion") mais choisit quand même une classe ferme au lieu de dire "uncertain". C'est le vrai problème à corriger : forcer "uncertain" quand le texte montre du doute.

Les 2 autres erreurs : un vrai faux positif, et un cas où le modèle a été prudent à raison (pas un vrai problème).

20 images c'est petit pour juger un score. Le but ici n'est pas d'avoir 100%, mais de montrer qu'on comprend pourquoi le modèle se trompe.